In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from binance.client import Client
from matplotlib import pyplot as plt

# import aggtrades
cwd = Path(__file__).resolve()
file_path = cwd.parent/'__input_ag'/'SOLUSDT-aggTrades-2026-06-20.csv'
# load csv to df
df = pd.read_csv(file_path)
# rename columns

# df = df.rename(columns={'a': 'agg_tde_id', 'p': 'price', 'q': 'base_qty',
#                         'f': 'first_tde_id', 'l': 'last_tde_id',
#                         'T': 'transact_time', 'm': 'is_buyer_maker', 'M': 'ignore'})
df = df.rename(columns={'agg_trade_id': 'agg_tde_id', 'quantity': 'base_qty',
                        'first_trade_id': 'first_tde_id', 'last_trade_id': 'last_tde_id'
                        })

# format some columns -- keep time as datetime , not string for GROUPING
# .dt.strftime("%Y-%m-%d %H:%M:%S")
df['transact_time'] = pd.to_datetime(df['transact_time'], unit='ms')
# helper columns
df['quote_qty_usdt'] = df['price'] * df['base_qty']
df['buy_vol_usdt'] = np.where(
    df['is_buyer_maker'] == False, df['quote_qty_usdt'], 0.0)
df['sell_vol_usdt'] = np.where(
    df['is_buyer_maker'] == True, df['quote_qty_usdt'], 0.0)

# BUCKETING TIME WINDOWS --- groupby --- aggregating
bdf = df.groupby(pd.Grouper(key='transact_time', freq='5Min')).agg(tde1_price=('price', 'first'),
                                                                   last_price=(
                                                                       'price', 'last'),
                                                                   buyVol_usdt=(
                                                                       'buy_vol_usdt', 'sum'),
                                                                   sellVol_usdt=(
                                                                       'sell_vol_usdt', 'sum')
                                                                   ).reset_index()
# rename columns
bdf = bdf.rename(columns={'transact_time': 'time'})
# derived columns
bdf['delta'] = bdf['buyVol_usdt'] - bdf['sellVol_usdt']
bdf['prc_%change_window'] = (
    bdf['last_price']-bdf['tde1_price'])/bdf['tde1_price']
day_open_price = bdf['tde1_price'].iloc[0]
bdf['dayOpen_prc%change'] = (bdf['last_price'] - day_open_price)/day_open_price

# column naming convention
# 1 time = the time duration of the stamp
# 2 tde1_price = the first trade price in the time window
# 3 last_price = the price of the last trade in the time window
# 4 buyVol_usdt = aggressive buy volume in usdt
# 5 delta
# 6 sellVol_usdt = aggressive sell volume in usdt
# 7 prc_%change_window = the price percetange change in time window
# 8 dayOpen_prc%change = is the overall price percentage change from day open

# ------------ GRAPHS ANALYSIS  -------------------------------
plt.plot( bdf['time'], bdf['buyVol_usdt'], label ="aggressive buy volume")
plt.plot( bdf['time'], bdf['sellVol_usdt'], label="aggressive sell volume")
plt.title("volumes against time")
plt.xlabel("time to the day")
plt.ylabel('volumes')
plt.grid()
plt.legend()
plt.show()